In [1]:
!pip install kaggle

In [7]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = "/content"

In [3]:
!kaggle datasets download -d ananaymital/us-used-cars-dataset

Dataset URL: https://www.kaggle.com/datasets/ananaymital/us-used-cars-dataset
License(s): copyright-authors
 99% 2.11G/2.13G [00:23<00:00, 149MB/s]
100% 2.13G/2.13G [00:23<00:00, 96.1MB/s]


In [4]:
!unzip /content/us-used-cars-dataset.zip -d /content

Archive:  /content/us-used-cars-dataset.zip
  inflating: /content/used_cars_data.csv  


In [2]:
import pandas as pd

In [10]:
# Check the file size of the dataset
file_path = '/content/used_cars_data.csv'
file_size = os.path.getsize(file_path) / (1024 * 1024 * 1024)  # Convert size to GB
print(f"Dataset size: {file_size:.2f} GB")

Dataset size: 9.29 GB


In [12]:
# Target size in bytes
target_size_gb = 2.1  # 2.1 GB

# Calculate the proportion of data to sample
proportion = target_size_gb / file_size
print(f"Sampling proportion: {proportion:.4f}")

Sampling proportion: 0.2259


In [14]:
# Read data in chunks and sample
chunk_size = 500000  # Adjust chunk size as needed
chunk_list = []

# Read data in chunks with low_memory=False
for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
    sampled_chunk = chunk.sample(frac=proportion)
    chunk_list.append(sampled_chunk)

In [15]:
sampled_df = pd.concat(chunk_list, axis=0)
print(f"Sampled DataFrame shape: {sampled_df.shape}")

Sampled DataFrame shape: (677805, 66)


In [16]:
sampled_df.to_csv('/content/sampled_used_cars.csv', index=False)

In [17]:
file_path = '/content/sampled_used_cars.csv'
df = pd.read_csv(file_path)

<ipython-input-17-3f2921212e47>:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [19]:
pip install dask[dataframe]

INFO: pip is looking at multiple versions of dask-expr to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.2/243.2 kB 12.6 MB/s eta 0:00:00


In [23]:
import dask.dataframe as dd

dtype_dict = {
    'bed': 'object',
    'cabin': 'object',
    'dealer_zip': 'object'
}

# Read with Dask
dask_df = dd.read_csv('/content/sampled_used_cars.csv', dtype=dtype_dict)
dask_result = dask_df.compute()

In [25]:
os.environ['MODIN_MEMORY'] = '4GB'

In [26]:
import modin.pandas as mpd

# Read with Modin
modin_df = mpd.read_csv(file_path)

OutOfMemoryError: Task was killed due to the node running low on memory.
Memory on the node (IP: 172.28.0.12, ID: 8a7f011bd88395a4a6d8c4e6c278c4f74b0c7ef96a5df5e3f4c47578) where the task (task ID: f8ecaf04b59419cea33017bd7a025d417e1d8e4501000000, name=_deploy_ray_func, pid=39521, memory used=1.61GB) was running was 12.08GB / 12.67GB (0.952973), which exceeds the memory usage threshold of 0.95. Ray killed this worker (ID: fe4dbc12470c9527ab7470db52a9b1e416d5e968decfa8bf79447c92) because it was the most recently scheduled task; to see more information about memory usage on this node, use `ray logs raylet.out -ip 172.28.0.12`. To see the logs of the worker, use `ray logs worker-fe4dbc12470c9527ab7470db52a9b1e416d5e968decfa8bf79447c92*out -ip 172.28.0.12. Top 10 memory users:
PID	MEM(GB)	COMMAND
28844	8.77	/usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-7ca6c552-e944...
39521	1.61	
37974	0.29	node /datalab/web/pyright/pyright-langserver.js --stdio --c

In [27]:
import ray
import ray.data

# Read with Ray
ray.init(ignore_reinit_error=True)
ray_df = ray.data.read_csv(file_path)

2025-02-12 13:33:23,358	INFO worker.py:1672 -- Calling ray.init() again after it has already been called.


In [28]:
# Perform a simple operation
ray_df = ray_df.map(lambda x: x)

In [29]:
# Clean column names by removing spaces and special characters
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace(r'\W', '', regex=True).str.lower()

In [30]:
# Check cleaned column names
print(f"Cleaned Columns: {df.columns.tolist()}")

Cleaned Columns: ['vin', 'back_legroom', 'bed', 'bed_height', 'bed_length', 'body_type', 'cabin', 'city', 'city_fuel_economy', 'combine_fuel_economy', 'daysonmarket', 'dealer_zip', 'description', 'engine_cylinders', 'engine_displacement', 'engine_type', 'exterior_color', 'fleet', 'frame_damaged', 'franchise_dealer', 'franchise_make', 'front_legroom', 'fuel_tank_volume', 'fuel_type', 'has_accidents', 'height', 'highway_fuel_economy', 'horsepower', 'interior_color', 'iscab', 'is_certified', 'is_cpo', 'is_new', 'is_oemcpo', 'latitude', 'length', 'listed_date', 'listing_color', 'listing_id', 'longitude', 'main_picture_url', 'major_options', 'make_name', 'maximum_seating', 'mileage', 'model_name', 'owner_count', 'power', 'price', 'salvage', 'savings_amount', 'seller_rating', 'sp_id', 'sp_name', 'theft_title', 'torque', 'transmission', 'transmission_display', 'trimid', 'trim_name', 'vehicle_damage_category', 'wheel_system', 'wheel_system_display', 'wheelbase', 'width', 'year']


In [31]:
import yaml

# Create a dictionary of the columns
columns_dict = {"columns": df.columns.tolist()}

# Write to a YAML file
with open('/content/columns_schema.yml', 'w') as yaml_file:
    yaml.dump(columns_dict, yaml_file)

In [32]:
# Load the schema from the YAML file
with open('/content/columns_schema.yml', 'r') as yaml_file:
    schema = yaml.safe_load(yaml_file)

# Validate the columns
schema_columns = set(schema['columns'])
df_columns = set(df.columns)

if schema_columns == df_columns:
    print("The columns in the dataset match")
else:
    print(f"Columns do not match. Dataset columns: {df_columns}, Schema columns: {schema_columns}")

The columns in the dataset match


In [33]:
# Write to a pipe-separated (|) file in GZ format
df.to_csv('/content/sampled_used_cars_pipe_separated.gz', sep='|', compression='gzip', index=False)

In [34]:
# Summary of the dataset
total_rows = df.shape[0]
total_columns = df.shape[1]
file_size_gb = os.path.getsize('/content/sampled_used_cars_pipe_separated.gz') / (1024 * 1024 * 1024)

print(f"Total number of rows: {total_rows}")
print(f"Total number of columns: {total_columns}")
print(f"File size (in GB): {file_size_gb:.2f}")

Total number of rows: 677805
Total number of columns: 66
File size (in GB): 0.61
